# Books: scrape, enrich, frame

The exploratory notebook behind this project, kept with its original outputs. It scrapes the first five catalogue pages of books.toscrape.com, looks up the first 20 titles in the Open Library Search API, and builds the typed silver table and two aggregates.

The same logic, split into tested modules, lives in `src/books_pipeline/`; use `python -m books_pipeline` for the full 1,000-book run. Running this notebook top to bottom makes live requests (5 catalogue pages and 20 Open Library lookups, one per second) and writes into `../data/`. Set `BOOKS_SCRAPER_CONTACT` to your email first so the site can reach you.


In [1]:
import os, time, json, re
from pathlib import Path
from urllib.parse import urljoin
import requests, pandas as pd
from bs4 import BeautifulSoup

ROOT = Path("..")
for sub in ("data/raw", "data/api"):
    (ROOT / sub).mkdir(parents=True, exist_ok=True)

START_URL = "http://books.toscrape.com/catalogue/page-1.html"
CONTACT   = os.environ.get("BOOKS_SCRAPER_CONTACT")
HEADERS   = {"User-Agent": f"books-scrape-to-silver/0.1 ({CONTACT})" if CONTACT else "books-scrape-to-silver (portfolio project)"}
PAUSE     = 1.0                                                                             # one request per second

def get(url: str) -> requests.Response | None:
    """One polite GET. Returns None (and prints why) instead of crashing on a bad page."""
    time.sleep(PAUSE)
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
        r.encoding = "utf-8"         # the server sends no charset, so requests would guess ISO-8859-1 and turn £ into Â£
        return r
    except requests.RequestException as e:
        print(f"skipped {url}: {e}")
        return None

## 1 · Scrape five catalogue pages

Follow the “next” link instead of typing page numbers, saving each page under `data/raw` (bronze).

In [2]:
def parse_page(html: str) -> list[dict]:
    """All books on one catalogue page: title, price, rating, availability, url (strings)."""
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for pod in soup.select("article.product_pod"):
        rows.append({
            "title":        pod.h3.a["title"],
            "price":        pod.select_one("p.price_color").get_text(strip=True),
            "rating":       pod.select_one("p.star-rating")["class"][1],  
            "availability": pod.select_one("p.instock.availability").get_text(strip=True),
            "url":          urljoin(START_URL, pod.h3.a["href"]),
        })
    return rows

def next_url(html: str, page_url: str) -> str | None:
    """Absolute URL of the next catalogue page, or None when there isn't one."""
    soup = BeautifulSoup(html, "html.parser")
    nxt = soup.select_one("li.next a")
    return urljoin(page_url, nxt["href"]) if nxt else None

def scrape(start_url: str, max_pages: int = 5) -> pd.DataFrame:
    rows, url, n = [], start_url, 0
    while url and n < max_pages:
        r = get(url)
        if r is None:
            break                    # a bad page ends the walk politely instead of crashing
        n += 1
        (ROOT / "data/raw" / f"page-{n}.html").write_text(r.text, encoding="utf-8")   # Bronze
        page_rows = parse_page(r.text)
        for row in page_rows:
            row["page"] = n
        rows.extend(page_rows)
        print(f"page {n}: {len(page_rows)} books")
        url = next_url(r.text, url)
    return pd.DataFrame(rows)

books_raw = scrape(START_URL, max_pages=5)
books_raw.head()
display(books_raw)

page 1: 20 books
page 2: 20 books
page 3: 20 books
page 4: 20 books
page 5: 20 books


,title,price,rating,availability,url,page
0,A Light in the Attic,£51.77,Three,In stock,http://books.toscrape.com/catalogue/a-light-in...,1
1,Tipping the Velvet,£53.74,One,In stock,http://books.toscrape.com/catalogue/tipping-th...,1
2,Soumission,£50.10,One,In stock,http://books.toscrape.com/catalogue/soumission...,1
3,Sharp Objects,£47.82,Four,In stock,http://books.toscrape.com/catalogue/sharp-obje...,1
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock,http://books.toscrape.com/catalogue/sapiens-a-...,1
...,...,...,...,...,...,...
95,Lumberjanes Vol. 3: A Terrible Plan (Lumberjan...,£19.92,Two,In stock,http://books.toscrape.com/catalogue/lumberjane...,5
96,"Layered: Baking, Building, and Styling Spectac...",£40.11,One,In stock,http://books.toscrape.com/catalogue/layered-ba...,5
97,Judo: Seven Steps to Black Belt (an Introducto...,£53.90,Two,In stock,http://books.toscrape.com/catalogue/judo-seven...,5
98,Join,£35.67,Five,In stock,http://books.toscrape.com/catalogue/join_902/i...,5


In [3]:
# Check: 100 rows, 6 columns
assert len(books_raw) == 100, len(books_raw)
books_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         100 non-null    object
 1   price         100 non-null    object
 2   rating        100 non-null    object
 3   availability  100 non-null    object
 4   url           100 non-null    object
 5   page          100 non-null    int64 
dtypes: int64(1), object(5)
memory usage: 4.8+ KB


In [4]:
# What a bad page does: get() logs it and returns None instead of crashing
get("http://books.toscrape.com/catalogue/page-999.html")

skipped http://books.toscrape.com/catalogue/page-999.html: 404 Client Error: Not Found for url: http://books.toscrape.com/catalogue/page-999.html


## 2 · Enrich from Open Library

Open Library Search API (https://openlibrary.org/dev/docs/api/search). Every JSON response is saved under `data/api/`.

In [5]:
OL = "https://openlibrary.org/search.json"
OL_HEADERS = HEADERS
FIELDS = "title,author_name,first_publish_year,isbn,subject,edition_count"

def safe_name(title: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", title.lower())[:80]

def lookup_title(title: str) -> dict:
    """Best match for a title, flattened. Always returns a dict with our title; other fields None if no match."""
    time.sleep(PAUSE)
    params = {"title": title, "limit": 1, "fields": FIELDS}
    resp = requests.get(OL, params=params, headers=OL_HEADERS, timeout=20)
    resp.raise_for_status()
    payload = resp.json()
    (ROOT / "data/api" / f"{safe_name(title)}.json").write_text(json.dumps(payload), encoding="utf-8")
    docs = payload.get("docs") or []
    d = docs[0] if docs else {}
    return {
        "title":              title,
        "ol_title":           d.get("title"),
        "author":             (d.get("author_name") or [None])[0],
        "first_publish_year": d.get("first_publish_year"),
        "edition_count":      d.get("edition_count"),
        "n_subjects": len(d.get("subject") or []) if docs else None,
    }

ol = pd.DataFrame([lookup_title(t) for t in books_raw["title"].head(20)])
ol

,title,ol_title,author,first_publish_year,edition_count,n_subjects
0,A Light in the Attic,A Light in the Attic,Shel Silverstein,1981.0,27.0,25.0
1,Tipping the Velvet,Tipping the Velvet,Sarah Waters,1998.0,22.0,24.0
2,Soumission,Soumission,Michel Houellebecq,2015.0,21.0,10.0
3,Sharp Objects,Sharp Objects,Gillian Flynn,2006.0,31.0,26.0
4,Sapiens: A Brief History of Humankind,Summary of Sapiens : a Brief History of Humankind,BookNation,2020.0,1.0,0.0
5,The Requiem Red,The Requiem Red,Brynn Chapman,2016.0,2.0,18.0
6,The Dirty Little Secrets of Getting Your Dream...,The dirty little secrets of getting your dream...,Don Raskin,2016.0,2.0,4.0
7,The Coming Woman: A Novel Based on the Life of...,None,None,NaN,NaN,NaN
8,The Boys in the Boat: Nine Americans and Their...,Summary of the Boys in the Boat : Nine America...,BookNation,2020.0,1.0,0.0
9,The Black Maria,The black Maria,Aracelis Girmay,2016.0,1.0,16.0


In [6]:
# Check: 20 rows (some unmatched), 20 JSON files saved
assert len(ol) == 20 and len(list((ROOT / "data/api").glob("*.json"))) >= 20
print(f"{ol['ol_title'].notna().sum()} of 20 matched")

15 of 20 matched


**Hand check of five matches.**

I checked 5 books against the responses saved in `data/api`. Here is what I saw: two are the right book (A Light in the Attic by Shel Silverstein; Sharp Objects by Gillian Flynn), two are the wrong book (Sapiens matched a 2020 "Summary of Sapiens" by BookNation instead of Yuval Noah Harari's book, and Shakespeare's Sonnets matched a 1999 book credited to James Schiffer, not Shakespeare), and one cannot be confirmed (Set Me Free by Denise Robins which is one of 120 Open Library results for that title, and the shop page gives no author to compare against). Matching on title alone accepts whatever Open Library ranks first, so study guides, books about the book and unrelated books that share a title all count as matches, and their `first_publish_year`, `edition_count` and `decade` then feed the averages as if they were correct.

## 3 · Type, join and summarise with pandas

In [7]:
RATING = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
books = books_raw.copy()
print("BEFORE"); books.info()

books["price_gbp"]  = books["price"].str.replace("£", "").astype("float64")  
books["rating_int"] = books["rating"].map(RATING).astype("Int64")        
books["in_stock"]   = books["availability"].str.contains("In stock")
books["scraped_at"] = pd.Timestamp.utcnow()
books = books.drop_duplicates(subset=["title", "url"])

merged = books.merge(ol, on="title", how="left") 
merged["decade"] = (merged["first_publish_year"] // 10 * 10).astype("Int64")

by_rating = merged.groupby("rating_int").agg(n_books=("title", "count"), avg_price=("price_gbp", "mean")).round(2)
by_decade = merged.groupby("decade").agg(avg_price=("price_gbp", "mean")).round(2)  # matched books only: groupby skips missing decades

merged.to_csv(ROOT / "data" / "books_silver.csv", index=False)
merged.to_parquet(ROOT / "data" / "books_silver.parquet", index=False)
print("AFTER"); merged.info()
display(by_rating); display(by_decade)

BEFORE
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         100 non-null    object
 1   price         100 non-null    object
 2   rating        100 non-null    object
 3   availability  100 non-null    object
 4   url           100 non-null    object
 5   page          100 non-null    int64 
dtypes: int64(1), object(5)
memory usage: 4.8+ KB
AFTER
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   title               100 non-null    object             
 1   price               100 non-null    object             
 2   rating              100 non-null    object             
 3   availability        100 non-null    object             
 4   url                 100

,n_books,avg_price
rating_int,,
1,22,35.52
2,19,35.91
3,22,36.84
4,18,33.98
5,19,30.01


,avg_price
decade,
1980,51.77
1990,30.62
2000,41.42
2010,39.80
2020,38.42


In [8]:
# Check: dtypes changed, both files exist, no £ in the CSV price column
assert str(merged["price_gbp"].dtype) == "float64" and str(merged["rating_int"].dtype) == "Int64" and str(merged["in_stock"].dtype) == "bool"
assert (ROOT / "data" / "books_silver.csv").exists() and (ROOT / "data" / "books_silver.parquet").exists()
assert "£" not in pd.read_csv(ROOT / "data" / "books_silver.csv")["price_gbp"].astype(str).str.cat()

In [9]:
print("Books and average price by star rating")
display(
    by_rating.rename(columns={
        "n_books": "Number of books",
        "avg_price": "Average price (GBP)",
    })
)

Books and average price by star rating


,Number of books,Average price (GBP)
rating_int,,
1,22,35.52
2,19,35.91
3,22,36.84
4,18,33.98
5,19,30.01


In [10]:
print("Average price by first-publication decade")
display(
    by_decade.rename(columns={
        "avg_price": "Average price (GBP)",
    })
)

Average price by first-publication decade


,Average price (GBP)
decade,
1980,51.77
1990,30.62
2000,41.42
2010,39.80
2020,38.42


In [11]:
print("books_silver.csv")
display(merged.head(5))

books_silver.csv


,title,price,rating,availability,url,page,price_gbp,rating_int,in_stock,scraped_at,ol_title,author,first_publish_year,edition_count,n_subjects,decade
0,A Light in the Attic,£51.77,Three,In stock,http://books.toscrape.com/catalogue/a-light-in...,1,51.77,3,True,2026-10-05 04:46:20.932060+00:00,A Light in the Attic,Shel Silverstein,1981.0,27.0,25.0,1980
1,Tipping the Velvet,£53.74,One,In stock,http://books.toscrape.com/catalogue/tipping-th...,1,53.74,1,True,2026-10-05 04:46:20.932060+00:00,Tipping the Velvet,Sarah Waters,1998.0,22.0,24.0,1990
2,Soumission,£50.10,One,In stock,http://books.toscrape.com/catalogue/soumission...,1,50.10,1,True,2026-10-05 04:46:20.932060+00:00,Soumission,Michel Houellebecq,2015.0,21.0,10.0,2010
3,Sharp Objects,£47.82,Four,In stock,http://books.toscrape.com/catalogue/sharp-obje...,1,47.82,4,True,2026-10-05 04:46:20.932060+00:00,Sharp Objects,Gillian Flynn,2006.0,31.0,26.0,2000
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock,http://books.toscrape.com/catalogue/sapiens-a-...,1,54.23,5,True,2026-10-05 04:46:20.932060+00:00,Summary of Sapiens : a Brief History of Humankind,BookNation,2020.0,1.0,0.0,2020


#### What I learned
* Scraping: I learned how to use BeautifulSoup selectors to pull specific fields out of the HTML, and how following the site’s “next” link lets the scraper work its way through the catalogue one page at a time. Saving the HTML came in handy too, since it meant I could re-parse pages without downloading them all over again.

* APIs: Just because the API returns a result doesn’t mean it’s the right book. In my saved results, the Sapiens lookup came back as a BookNation summary instead of the actual book. That showed me how matching on the title alone can let the wrong authors and publication years slip into the dataset.
pandas: I learned how to convert the scraped text into types that were actually useful, like numeric prices, integer ratings, and a boolean for whether each book was in stock. Using a LEFT join meant all 100 scraped books stayed in the table, even though I only looked up the first 20 titles through the API.
Notebooks: I learned that editing a cell doesn’t update the output that’s already showing under it. That’s why restarting and running everything from the top really does matter. Otherwise, old results can stay on screen next to newer code.

* Claude: I learned to ask for evidence that generated code works instead of taking it at face value. The price test failed at first because of the incorrectly encoded £ signs. It passed once the saved HTML was fixed, and the test itself never had to change. Blocking requests during a cached rerun was also a good way to check that the scraper could run completely off the saved pages.

#### Where Claude needed help:
Claude wrote a lot of the extended scraper, but I still had to review its explanations and the assumptions it was making. Its first suggestion for the book-page selectors came from memory, so I asked it to check an actual page before using them. It also claimed that running the encoding repair again would fail safely, which turned out to be wrong. The repair we ended up with used backups and validation before replacing any files. I asked for download and parsing failures to be reported separately, and for a guard in case the catalogue came back empty. While reviewing the documentation, I also had to make sure the 1,000-row table in memory wasn’t confused with the 100-row silver files. I also had to stop the prompt log from getting overwritten. This made me realize that reviewing generated code is about more than whether the program finishes. It also means checking the data, how the code behaves when something fails, and the documentation.